# Kaggle Resume Orchestrator (CPU-only, multi-kernel)

**Purpose:** watches ALL of Sham's real Kaggle kernels (the GPU main track and every CPU track: continuous training, autonomous research, audio/image/video tokenizer tracks) and automatically pushes a fresh run for any one of them that has stopped (finished its own time budget, errored, or was cancelled) — so training resumes everywhere without anyone manually reopening each notebook and clicking "Save & Run All" again.

**Why ONE consolidated orchestrator instead of scheduling each notebook separately (owner request, 2026-09-21):** Kaggle caps scheduled notebooks at 5 private / 15 public. Six-plus real training tracks would either blow through that cap or force making some of them public (exposing real training code/architecture unnecessarily). This orchestrator collapses ALL of them into **one single scheduled slot** — it never trains anything itself, it only checks each real kernel's status and re-triggers whichever ones have stopped. Every underlying training notebook stays completely unscheduled and fully private; only this one orchestrator needs "Schedule this notebook to run."

**Why this is a SEPARATE, CPU-only notebook rather than scheduling the GPU training notebook itself:** Kaggle's own native "Schedule this notebook" feature refuses to schedule any notebook with a GPU accelerator attached. This one has **no accelerator at all** — set it to `None` when you configure it — specifically so Kaggle's own scheduler accepts it.

**Revision note (2026-09-20):** an earlier attempt drove this same check-and-resume logic from a GitHub Actions workflow instead. That failed in practice (the push did not reliably take effect and training then failed at startup) and has been removed. This notebook reuses the exact same, already-tested `resume_kernel()` decision logic — see `ai-system/scripts/kaggle_auto_resume.py`'s own module docstring and self-test — just called from inside Kaggle's own infrastructure instead of from GitHub, now looped over every real target kernel instead of just one.

**One-time setup, before scheduling this:**
1. Set the accelerator for THIS notebook to **None**. Turn **Internet On**.
2. Attach the same 3 secrets already used by the training notebooks (Add-ons → Secrets): `GITHUB_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY`.
3. Fill in `TARGET_KERNELS` below with every real kernel slug you actually run (shown in each notebook's own Kaggle URL, e.g. `jonsnowjonsnow/notebook2d0e40c1f1`) — leave out any track you are not currently running.
4. Save this notebook, then use Kaggle's own **"Schedule this notebook"** button (Settings) — e.g. every 1-2 hours, so a stopped kernel is picked back up promptly regardless of which track's own time budget just ran out.
5. **Make sure none of the individual training notebooks are ALSO separately scheduled** — this orchestrator replaces that, not adds to it. Two schedulers resuming the same kernel independently could double-push it.

**Known platform caveat** (unchanged from before, not something this notebook can fix): a `kaggle kernels push` can disconnect the pushed kernel's own attached Secrets even when it reports success. Spot-check each training notebook's own Secrets tab after the first few automated resumes to confirm they held.

In [ ]:
# EDIT THIS before scheduling -- every entry is a real kernel slug this
# orchestrator watches and resumes; leave out any track you are not
# currently running. Never left as a guess -- a wrong slug would
# resume/duplicate the wrong kernel entirely.
TARGET_KERNELS = [
    "REPLACE_ME/main-gpu-training-kernel-slug",       # المسار الرئيسي -- sham-checkpoint (GPU)
    # "REPLACE_ME/cpu-training-track-kernel-slug",    # Track A -- التدريب المستمر على CPU
    # "REPLACE_ME/cpu-research-track-kernel-slug",    # Track B -- البحث الذاتي المستمر
    # "REPLACE_ME/audio-tokenizer-track-kernel-slug", # مسار ترميز الصوت
    # "REPLACE_ME/image-tokenizer-track-kernel-slug", # مسار ترميز الصورة
    # "REPLACE_ME/video-tokenizer-track-kernel-slug", # مسار جمع/ترميز الفيديو
]

assert TARGET_KERNELS, "أضيفي كيان (slug) واحداً حقيقياً على الأقل إلى TARGET_KERNELS أعلاه قبل الجدولة."
assert all(not slug.startswith("REPLACE_ME") for slug in TARGET_KERNELS), (
    "لا يزال أحد الإدخالات في TARGET_KERNELS يحمل القيمة النائبة REPLACE_ME -- "
    "استبدليها بالـ slug الحقيقي لكل دفتر تريدين مراقبته (يظهر في رابط الدفتر نفسه على Kaggle)، "
    "أو احذفي (علّقي) السطر إن كنتِ لا تشغّلين ذلك المسار حالياً."
)


In [ ]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

# Same clone/pull pattern as sham_small_training.ipynb's own first cell
# -- this notebook always runs against the LATEST kaggle_auto_resume.py
# in the repo, never a stale copy pasted in here by hand.
GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=True)
print("repo ready, kaggle package installed.")

In [ ]:
import os
import sys
from kaggle_secrets import UserSecretsClient

# Same Kaggle-Secrets pattern already used by every training notebook's
# own checkpoint-publish cell -- no new secret needed, this reuses the
# exact two that already exist for that purpose.
KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

sys.path.insert(0, "/kaggle/working/Ttbik/ai-system/scripts")
from kaggle_auto_resume import _load_kaggle_api, resume_kernel

# One authenticated api object reused across every kernel this run --
# no reason to re-authenticate per target.
api = _load_kaggle_api()

results = {}
for kernel_slug in TARGET_KERNELS:
    print(f"\n=== checking {kernel_slug} ===")
    try:
        # sync_from_repo=False deliberately, same reasoning as the
        # original single-kernel version: a real trained notebook's LIVE
        # Kaggle cells may hold manual edits (calibration constants,
        # hyperparameters) made directly in Kaggle's own editor that are
        # NOT necessarily mirrored back into this git repo's reference
        # copy -- overwriting from the repo here would silently discard
        # those (a real, owner-hit case, 2026-09-21: MAX_TRAINING_HOURS
        # edited live on Kaggle differed from the repo's own value).
        results[kernel_slug] = resume_kernel(
            kernel_slug=kernel_slug,
            notebook_repo_path=None,
            work_dir=f"/kaggle/working/kaggle-kernel/{kernel_slug.replace('/', '_')}",
            sync_from_repo=False,
            api=api,
        )
    except Exception as exc:
        print(f"kaggle_auto_resume: unexpected error resuming '{kernel_slug}' ({exc}) -- "
              f"leaving it as-is rather than risking a bad push; check it manually.")
        results[kernel_slug] = f"error: {exc}"

print("\n=== summary ===")
for kernel_slug, outcome in results.items():
    print(f"{kernel_slug}: {outcome}")
